# Calculus for AI and Mechanistic Interpretability

This notebook is a detailed calculus guide designed for deep learning, optimization, and interpretability research.

## Why calculus matters

If linear algebra is the language of representations, calculus is the language of **change**.

Calculus tells you:

- how outputs change when inputs change
- how loss changes when parameters change
- how to optimize a model
- how sensitivity propagates through layers
- how gradients encode local information

In deep learning, backpropagation is essentially a large, structured application of the **chain rule**.

## What this notebook covers

1. Functions and rates of change
2. Limits and continuity
3. Derivatives
4. Common derivative rules
5. Partial derivatives
6. Gradients
7. Jacobians
8. Hessians
9. Chain rule
10. Taylor approximation
11. Optimization and gradient descent
12. Calculus in neural networks
13. Autograd in PyTorch
14. Exercises

In [ ]:
import numpy as np
import torch

np.set_printoptions(suppress=True, precision=5)
torch.set_printoptions(precision=5, sci_mode=False)

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)

## 1. Functions and rates of change

A function maps input values to output values:

$$
f: \mathbb{R} \to \mathbb{R}
$$

Example:

$$
f(x) = x^2
$$

The key calculus question is:

> If $x$ changes a little, how much does $f(x)$ change?

That local rate of change is captured by the **derivative**.

### Why this matters in AI

A model is a function:

$$
f_\theta(x)
$$

where:
- $x$ is input
- $\theta$ are parameters

Training asks:

> How should we change $\theta$ to reduce loss?

That is a calculus question.

## 2. Limits

The derivative is built from the concept of a limit.

We write:

$$
\lim_{x \to a} f(x) = L
$$

to mean that as $x$ gets arbitrarily close to $a$, the function gets arbitrarily close to $L$.

### Example

For $f(x)=x^2$,

$$
\lim_{x \to 2} x^2 = 4
$$

### Why limits matter

They let us define instantaneous change rather than average change.

## 3. Derivative: formal definition

The derivative of $f(x)$ at $x$ is:

$$
f'(x) = \lim_{h \to 0}\frac{f(x+h)-f(x)}{h}
$$

This is the limit of the average rate of change.

### Geometric meaning

The derivative is the slope of the tangent line at that point.

### Example: derivative of $x^2$

Let:

$$
f(x)=x^2
$$

Then:

$$
\frac{f(x+h)-f(x)}{h}
=
\frac{(x+h)^2 - x^2}{h}
=
\frac{x^2 + 2xh + h^2 - x^2}{h}
=
\frac{2xh + h^2}{h}
=
2x + h
$$

Taking the limit as $h \to 0$:

$$
f'(x)=2x
$$

In [ ]:
def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x)) / h

f = lambda x: x**2

for x in [0.0, 1.0, 2.0, 3.0]:
    print(f"x={x: .1f}, numerical derivative={numerical_derivative(f, x): .5f}, exact={2*x: .5f}")

## 4. Common derivative rules

You should memorize the most important derivative patterns.

### Constant rule

$$
\frac{d}{dx} c = 0
$$

### Power rule

$$
\frac{d}{dx} x^n = n x^{n-1}
$$

### Constant multiple rule

$$
\frac{d}{dx}[c f(x)] = c f'(x)
$$

### Sum rule

$$
\frac{d}{dx}[f(x)+g(x)] = f'(x)+g'(x)
$$

### Product rule

$$
\frac{d}{dx}[f(x)g(x)] = f'(x)g(x) + f(x)g'(x)
$$

### Quotient rule

$$
\frac{d}{dx}\left[\frac{f(x)}{g(x)}\right]
=
\frac{f'(x)g(x)-f(x)g'(x)}{g(x)^2}
$$

### Chain rule

$$
\frac{d}{dx} f(g(x)) = f'(g(x)) g'(x)
$$

This last one is the backbone of backpropagation.

## 5. Important derivatives for ML

These show up repeatedly in AI:

### Exponential
$$
\frac{d}{dx} e^x = e^x
$$

### Natural logarithm
$$
\frac{d}{dx}\log x = \frac{1}{x}
$$

### Sigmoid
$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

Derivative:
$$
\sigma'(x)=\sigma(x)(1-\sigma(x))
$$

### Hyperbolic tangent
$$
\tanh'(x)=1-\tanh^2(x)
$$

### ReLU
$$
\mathrm{ReLU}(x)=\max(0,x)
$$

Derivative:
$$
\mathrm{ReLU}'(x)=
\begin{cases}
1 & x>0 \\
0 & x<0
\end{cases}
$$

At $x=0$, the derivative is not uniquely defined, but implementations choose a convention.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

xs = np.array([-3., -1., 0., 1., 3.])
print("x      =", xs)
print("sigmoid=", sigmoid(xs))
print("sig'   =", sigmoid_derivative(xs))
print("relu   =", np.maximum(0, xs))

## 6. Partial derivatives

When a function depends on several variables, we use **partial derivatives**.

Suppose:

$$
f(x,y) = x^2 y + 3y
$$

Then:

### Partial derivative with respect to $x$

Treat $y$ as constant:

$$
\frac{\partial f}{\partial x} = 2xy
$$

### Partial derivative with respect to $y$

Treat $x$ as constant:

$$
\frac{\partial f}{\partial y} = x^2 + 3
$$

### Why this matters in AI

A neural network loss depends on many parameters:

$$
L(w_1, w_2, \dots, w_n)
$$

Training needs the partial derivative of the loss with respect to **each parameter**.

In [ ]:
def f(x, y):
    return x**2 * y + 3*y

x, y = 2.0, 4.0

df_dx = 2*x*y
df_dy = x**2 + 3

print("f(x, y) =", f(x, y))
print("∂f/∂x =", df_dx)
print("∂f/∂y =", df_dy)

## 7. Gradient

For a scalar-valued function of many variables:

$$
f(\mathbf{x}) = f(x_1, x_2, \dots, x_n)
$$

the **gradient** is the vector of partial derivatives:

$$
\nabla f(\mathbf{x})
=
\begin{bmatrix}
\frac{\partial f}{\partial x_1} \\
\frac{\partial f}{\partial x_2} \\
\vdots \\
\frac{\partial f}{\partial x_n}
\end{bmatrix}
$$

### Interpretation

The gradient points in the direction of steepest increase.

So the negative gradient points in the direction of steepest decrease.

### Why this matters

Gradient descent uses:

$$
\theta_{\text{new}} = \theta - \eta \nabla_\theta L
$$

where $\eta$ is the learning rate.

In [ ]:
def f_grad(x):
    # f(x1, x2) = x1^2 + 3*x2^2
    x1, x2 = x
    value = x1**2 + 3*x2**2
    grad = np.array([2*x1, 6*x2])
    return value, grad

point = np.array([2.0, -1.0])
value, grad = f_grad(point)

print("Point:", point)
print("f(point) =", value)
print("Gradient =", grad)

## 8. Directional derivative

The directional derivative measures the rate of change of $f$ in a particular direction $\mathbf{u}$:

$$
D_{\mathbf{u}} f(\mathbf{x}) = \nabla f(\mathbf{x})^\top \mathbf{u}
$$

assuming $\mathbf{u}$ is a unit vector.

### Interpretability connection

If you have a meaningful direction in activation space, the directional derivative tells you how a scalar quantity changes when moving in that direction.

For example:
- how does a logit change if we move along a feature direction?
- how does truth score change if we move along a truth direction?

In [ ]:
grad = np.array([4., -6.])  # from previous example at point [2, -1]
u = np.array([1., 1.])
u = u / np.linalg.norm(u)

directional_derivative = grad @ u
print("Gradient =", grad)
print("Unit direction =", u)
print("Directional derivative =", directional_derivative)

## 9. Jacobian

If a function maps vectors to vectors:

$$
\mathbf{f}: \mathbb{R}^n \to \mathbb{R}^m
$$

then the **Jacobian** collects all first derivatives:

$$
J_{ij} = \frac{\partial f_i}{\partial x_j}
$$

So the Jacobian is an $m \times n$ matrix.

### Why Jacobians matter

They describe local linear behavior of a vector-valued function.

In deep learning, each layer can be locally approximated by a Jacobian.

### Example

If

$$
\mathbf{f}(x,y) =
\begin{bmatrix}
x^2 + y \\
xy
\end{bmatrix}
$$

then

$$
J(x,y)=
\begin{bmatrix}
2x & 1 \\
y & x
\end{bmatrix}
$$

In [ ]:
def jacobian_example(x, y):
    J = np.array([
        [2*x, 1.0],
        [y,   x]
    ])
    return J

x, y = 2.0, 3.0
print("Jacobian at (2, 3):\n", jacobian_example(x, y))

## 10. Hessian

For a scalar-valued function $f(\mathbf{x})$, the **Hessian** is the matrix of second derivatives:

$$
H_{ij} = \frac{\partial^2 f}{\partial x_i \partial x_j}
$$

### Why second derivatives matter

Second derivatives describe curvature.

- positive curvature: bowl-like
- negative curvature: hill-like
- mixed curvature: saddle behavior

### Why this matters in optimization

The Hessian helps distinguish:
- minima
- maxima
- saddle points

In deep learning, the Hessian is usually too expensive to work with exactly, but the concept is still important.

In [ ]:
def hessian_example(x1, x2):
    # f(x1, x2) = x1^2 + 3*x2^2 + x1*x2
    H = np.array([
        [2., 1.],
        [1., 6.]
    ])
    return H

print("Hessian:\n", hessian_example(0, 0))

## 11. Chain rule

This is the single most important rule for backpropagation.

If:

$$
y = f(g(x))
$$

then:

$$
\frac{dy}{dx} = \frac{dy}{dg}\frac{dg}{dx}
$$

More explicitly:

$$
\frac{d}{dx} f(g(x)) = f'(g(x))g'(x)
$$

### Example

Let:

$$
g(x) = x^2,\quad f(u)=\sin u
$$

Then:

$$
y = f(g(x)) = \sin(x^2)
$$

Derivative:

$$
\frac{dy}{dx} = \cos(x^2)\cdot 2x
$$

### Why this matters in neural networks

A network is a composition of many functions:

$$
x \to h_1 \to h_2 \to \cdots \to \hat{y} \to L
$$

Backpropagation repeatedly applies the chain rule backward through the computation graph.

In [ ]:
def y(x):
    return np.sin(x**2)

def dy_dx_exact(x):
    return np.cos(x**2) * 2*x

for x in [0.5, 1.0, 2.0]:
    num = (y(x + 1e-5) - y(x)) / 1e-5
    exact = dy_dx_exact(x)
    print(f"x={x}, numerical={num:.5f}, exact={exact:.5f}")

## 12. Multivariable chain rule and backprop intuition

Suppose:

$$
z = wx + b
$$

and

$$
\hat{y} = \sigma(z)
$$

and loss is

$$
L = \frac{1}{2}(\hat{y} - y)^2
$$

We want:

$$
\frac{\partial L}{\partial w}
$$

By chain rule:

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial z}
\frac{\partial z}{\partial w}
$$

Each piece is simpler:

$$
\frac{\partial L}{\partial \hat{y}} = \hat{y} - y
$$

$$
\frac{\partial \hat{y}}{\partial z} = \sigma(z)(1-\sigma(z))
$$

$$
\frac{\partial z}{\partial w} = x
$$

So:

$$
\frac{\partial L}{\partial w}
=
(\hat{y} - y)\sigma(z)(1-\sigma(z))x
$$

This is the basic flavor of backpropagation.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = 2.0
w = 0.7
b = -0.1
y_true = 1.0

z = w*x + b
y_hat = sigmoid(z)
L = 0.5 * (y_hat - y_true)**2

dL_dyhat = y_hat - y_true
dyhat_dz = y_hat * (1 - y_hat)
dz_dw = x

dL_dw = dL_dyhat * dyhat_dz * dz_dw

print("z =", z)
print("y_hat =", y_hat)
print("Loss =", L)
print("dL/dw =", dL_dw)

## 13. Taylor approximation

A first-order Taylor approximation of $f$ around point $a$ is:

$$
f(x) \approx f(a) + f'(a)(x-a)
$$

This says a smooth function is locally approximately linear.

For multivariable functions:

$$
f(\mathbf{x} + \Delta \mathbf{x})
\approx
f(\mathbf{x}) + \nabla f(\mathbf{x})^\top \Delta \mathbf{x}
$$

### Why this matters

Gradients tell you the best local linear approximation.

This is one reason gradients are so useful for optimization and sensitivity analysis.

In [ ]:
def f(x):
    return np.sin(x)

a = 0.5
for x in [0.4, 0.5, 0.6]:
    exact = f(x)
    approx = f(a) + np.cos(a) * (x - a)
    print(f"x={x:.1f}, exact={exact:.6f}, first-order approx={approx:.6f}")

## 14. Gradient descent

To minimize a differentiable function $f(\theta)$, gradient descent updates:

$$
\theta_{t+1} = \theta_t - \eta \nabla f(\theta_t)
$$

where:
- $\eta$ is the learning rate
- $\nabla f(\theta_t)$ is the gradient at current point

### Intuition

- gradient points uphill
- negative gradient points downhill
- step size controls how far you move

### Why this matters

This is the foundation of training neural networks.

In [ ]:
def f(x):
    return (x - 3)**2 + 1

def grad_f(x):
    return 2*(x - 3)

x = 10.0
lr = 0.1

trajectory = [x]
for _ in range(15):
    x = x - lr * grad_f(x)
    trajectory.append(x)

print("Gradient descent trajectory:")
print(np.array(trajectory))
print("Final value f(x) =", f(x))

## 15. Loss landscapes, minima, and saddle points

In one dimension, optimization looks simple. In many dimensions, the surface can contain:

- local minima
- local maxima
- saddle points
- flat regions

### Why saddle points matter

In high-dimensional problems, saddle points are often more common than bad local minima.

### Hessian intuition

At a critical point where $\nabla f = 0$:

- Hessian positive definite $\Rightarrow$ local minimum
- Hessian negative definite $\Rightarrow$ local maximum
- Hessian mixed signs $\Rightarrow$ saddle point

## 16. Calculus in neural networks

A neural network layer often computes:

$$
\mathbf{h} = \phi(W\mathbf{x} + \mathbf{b})
$$

Training minimizes a loss:

$$
L(\theta)
$$

where $\theta$ contains all weights and biases.

### What backpropagation does

Backpropagation computes gradients:

$$
\frac{\partial L}{\partial \theta}
$$

for all parameters efficiently by repeatedly applying the chain rule.

### Why gradients matter

They tell you:

- which direction reduces loss
- how sensitive outputs are to parameters
- how sensitive outputs are to activations
- how perturbations may change behavior

That last point is highly relevant to interpretability.

## 17. Input gradients and interpretability

Sometimes we care not about gradients with respect to weights, but with respect to inputs or hidden states.

Examples:

$$
\frac{\partial \text{logit}}{\partial \mathbf{x}}
\quad \text{or} \quad
\frac{\partial \text{score}}{\partial \mathbf{h}}
$$

These gradients can indicate:

- sensitivity
- saliency
- local directional effects

### Important caution

Gradients are local. They describe infinitesimal behavior around the current point. That can be useful, but not the whole story.

In [ ]:
# A tiny scalar example of sensitivity
def score(x1, x2):
    return x1**2 + 2*x1*x2

x1, x2 = 1.5, -0.5

dscore_dx1 = 2*x1 + 2*x2
dscore_dx2 = 2*x1

print("score =", score(x1, x2))
print("∂score/∂x1 =", dscore_dx1)
print("∂score/∂x2 =", dscore_dx2)

## 18. PyTorch autograd

PyTorch can compute gradients automatically.

This is extremely important for modern deep learning work.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x + 1

y.backward()

print("y =", y.item())
print("dy/dx =", x.grad.item())
print("Exact derivative at x=2 should be 2x + 3 =", 2*2 + 3)

## 19. Autograd with several variables

PyTorch can also compute partial derivatives for multivariable functions.

In [ ]:
x1 = torch.tensor(2.0, requires_grad=True)
x2 = torch.tensor(-1.0, requires_grad=True)

f = x1**2 + 3*x2**2 + x1*x2
f.backward()

print("f =", f.item())
print("∂f/∂x1 =", x1.grad.item())
print("∂f/∂x2 =", x2.grad.item())

## 20. A tiny one-neuron example in PyTorch

This connects calculus directly to a neural-network-style computation graph.

In [ ]:
x = torch.tensor(2.0)
y_true = torch.tensor(1.0)

w = torch.tensor(0.7, requires_grad=True)
b = torch.tensor(-0.1, requires_grad=True)

z = w * x + b
y_hat = torch.sigmoid(z)
loss = 0.5 * (y_hat - y_true)**2

loss.backward()

print("z =", z.item())
print("y_hat =", y_hat.item())
print("loss =", loss.item())
print("dloss/dw =", w.grad.item())
print("dloss/db =", b.grad.item())

## 21. Gradient descent in PyTorch

We can also perform manual optimization steps.

In [ ]:
# Minimize f(w) = (w - 5)^2
w = torch.tensor(0.0, requires_grad=True)
lr = 0.2

history = []

for step in range(10):
    loss = (w - 5)**2
    loss.backward()

    with torch.no_grad():
        w -= lr * w.grad

    history.append((step, w.item(), loss.item()))
    w.grad.zero_()

for step, w_val, loss_val in history:
    print(f"step={step:2d}, w={w_val:.5f}, loss={loss_val:.5f}")

## 22. Common pitfalls

### Pitfall 1: derivative is local, not global

A gradient only describes nearby change.

### Pitfall 2: zero gradient does not always mean minimum

It could be a maximum or saddle point.

### Pitfall 3: large learning rate can overshoot

Gradient descent needs a sensible step size.

### Pitfall 4: partial derivative vs total derivative

In multivariable settings, be careful about which variables are held fixed.

### Pitfall 5: gradients are not explanations by themselves

They are useful signals, but interpretability often needs more than saliency.

## 23. Summary

The most important calculus ideas for AI are:

- derivative = instantaneous rate of change
- partial derivative = sensitivity to one variable
- gradient = direction of steepest increase
- Jacobian = first-order behavior of vector-valued functions
- Hessian = curvature
- chain rule = backbone of backpropagation
- Taylor approximation = local linearization
- gradient descent = optimization by moving opposite the gradient

If you become comfortable with these, you will understand:

- backpropagation
- optimization
- sensitivity analysis
- local linear probes
- activation and parameter gradients
- many mechanistic interpretability techniques

# Exercises

## Conceptual

1. Explain in your own words why the gradient points in the direction of steepest increase.
2. Why is the chain rule essential for neural network training?
3. What is the difference between a Jacobian and a Hessian?
4. Why does a first-order Taylor approximation matter conceptually?
5. Why might gradients be useful but insufficient for interpretability?

## Computational

### Exercise 1: derivative by hand
For:
$$
f(x)=3x^3 - 2x + 5
$$
compute:
- derivative analytically
- numerical derivative at $x=2$

### Exercise 2: partial derivatives
For:
$$
f(x,y)=x^2y + y^3
$$
compute:
$$
\frac{\partial f}{\partial x}, \quad \frac{\partial f}{\partial y}
$$
and evaluate them at $(x,y)=(2,1)$.

### Exercise 3: gradient descent
Minimize:
$$
f(x)=(x-4)^2
$$
using gradient descent from $x=10$.

### Exercise 4: chain rule
Let:
$$
g(x)=x^2+1,\quad f(u)=\log u
$$
Form:
$$
y=f(g(x))=\log(x^2+1)
$$
Compute $\frac{dy}{dx}$ by hand and verify numerically.

### Exercise 5: PyTorch autograd
Use PyTorch to compute the gradient of:
$$
f(x_1, x_2)=x_1^2 + x_1x_2 + 2x_2^2
$$
at $(1,2)$.

### Exercise 6: sensitivity direction
Let:
$$
f(x_1,x_2)=x_1^2 + 3x_2^2
$$
At point $(1,-1)$, compute:
- gradient
- directional derivative in direction $(1,1)$ after normalization
Interpret the sign of the result.

In [ ]:
# Starter workspace for exercises

import numpy as np
import torch

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x)) / h

# Exercise 1
f1 = lambda x: 3*x**3 - 2*x + 5
x0 = 2.0
# exact = ...
# numerical = numerical_derivative(f1, x0)

# Exercise 2
x, y = 2.0, 1.0
# dfdx = ...
# dfdy = ...

# Exercise 3
def f3(x):
    return (x - 4)**2

def grad_f3(x):
    return 2*(x - 4)

# Run gradient descent here

# Exercise 4
f4 = lambda x: np.log(x**2 + 1)
# derivative by hand: 2x/(x^2+1)
# verify numerically

# Exercise 5
x1 = torch.tensor(1.0, requires_grad=True)
x2 = torch.tensor(2.0, requires_grad=True)
# f = ...
# f.backward()

# Exercise 6
point = np.array([1.0, -1.0])
# gradient = ...
# direction = np.array([1.0, 1.0])
# normalize and compute directional derivative

# What else in math you will eventually need

These three notebooks cover the core foundation. For your long-term goal in interpretability, the next math topics worth studying are:

1. **Information theory**
   - entropy
   - mutual information
   - KL divergence
   - cross-entropy

2. **Optimization**
   - SGD, momentum, Adam
   - convex vs non-convex optimization
   - Hessian intuition

3. **Matrix calculus**
   - derivatives with respect to vectors and matrices
   - very useful for reading papers cleanly

4. **Numerical methods**
   - floating point issues
   - stability
   - approximation

5. **Geometry**
   - manifolds, angles, projections, subspaces
   - especially useful for representation learning

But if you master:
- linear algebra
- probability/statistics
- calculus

you will already have the main mathematical foundation needed to go much deeper into AI and mechanistic interpretability.